# Volleyball Analytics — R&D Sandbox

Notatnik zsynchronizowany z `engine.py` (backend). Pozwala na testowanie i rozwijanie analityki **krok po kroku** na wybranej klatce wideo, niezależnie od aplikacji Flutter/FastAPI.

---

## Jak uzywac?
1. Uruchamiaj komórki **po kolei** (Shift+Enter)
2. Zmien sciezke wideo w **Kroku 2**
3. Eksperymentuj z parametrami (progi conf, liczba klatek itp.)
4. Po ulepszeniach wgraj poprawione wartosci do `engine.py`

Aby **zaktualizowac** ten notatnik do najnowszej wersji engine:
```bash
python sync_notebook.py --tag moj_eksperyment
```
Znajdziesz wtedy nowa wersje w: `notebooks/versions/`

---
## Krok 1: Importy i konfiguracja

In [ ]:
import cv2
import os
import sys
import math
import pickle
import numpy as np
import onnxruntime
import mediapipe as mp
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Dodaj sciezke backendu do importow (aby dzialal frame_utilities)
BACKEND_DIR = os.path.abspath("..")
if BACKEND_DIR not in sys.path:
    sys.path.insert(0, BACKEND_DIR)

from frame_utilities import (
    preprocess_yolo_input,
    postprocess_yolo_output,
    get_distance_person_ball_np,
    pad_frame_to_square
)

# Sciezka do folderu modeli
MODELS_DIR = os.path.join(BACKEND_DIR, "models")
print(f"Backend dir: {BACKEND_DIR}")
print(f"Models dir:  {MODELS_DIR}")
print(f"Dostepne modele: {os.listdir(MODELS_DIR)}")

---
## Krok 2: Inicjalizacja modeli

Laduje wszystkie modele tak jak w `VolleyballAnalyticsEngine.__init__()`.

In [ ]:
providers = ["DmlExecutionProvider", "CPUExecutionProvider"]

# --- YOLO: Detekcja osob (COCO) ---
session_coco = onnxruntime.InferenceSession(
    os.path.join(MODELS_DIR, "yolo11n.onnx"), providers=providers
)
input_name_coco  = session_coco.get_inputs()[0].name
output_name_coco = session_coco.get_outputs()[0].name
print("[OK] YOLO COCO (detekcja osob) zaladowany")

# --- YOLO: Detekcja pilki siatkowej ---
session_vb = onnxruntime.InferenceSession(
    os.path.join(MODELS_DIR, "yolo11n_vb.onnx"), providers=providers
)
input_name_vb  = session_vb.get_inputs()[0].name
output_name_vb = session_vb.get_outputs()[0].name
print("[OK] YOLO Volleyball (detekcja pilki) zaladowany")

# --- RandomForest: Klasyfikacja akcji ---
model_dict = pickle.load(open(os.path.join(MODELS_DIR, "model.p"), "rb"))
rf_model = model_dict["model"]
print(f"[OK] RandomForest zaladowany | Klasy: {rf_model.classes_}")

# --- MediaPipe: Estymacja pozy ---
mp_pose_solution = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils
mp_pose = mp_pose_solution.Pose(
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)
print("[OK] MediaPipe Pose zaladowany")

---
## Krok 3: Wybór wideo i parametry

> **Zmien `VIDEO_PATH`** na sciezke do wideo, ktore chcesz analizowac.

In [ ]:
# ============================================================
# USTAW TUTAJ SCIEZKE DO SWOJEGO WIDEO
VIDEO_PATH = r"C:\Users\kusoj\Desktop\Projekty\GoGoShawk\VideoMobile4Sport\ML_CV_Video\MLCV test.mp4"
# ============================================================

# Prog pewnosci detekcji — mozna zmieniac i testowac
CONF_BALL   = 0.5   # prog dla detekcji pilki
CONF_PERSON = 0.5   # prog dla detekcji osoby

cap = cv2.VideoCapture(VIDEO_PATH)
fps         = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print(f"Wideo:        {VIDEO_PATH}")
print(f"FPS:          {fps}")
print(f"Liczba klatek:{total_frames}")
print(f"Czas trwania: {total_frames/fps:.1f} s")

---
## Krok 4: Wczytanie klatki

Uruchamiaj te komórke wielokrotnie — za kazdym razem przechodzi do **kolejnej klatki**.  
Mozesz tez ustawic `FRAME_ID` i przeskoczyc do konkretnej klatki.

In [ ]:
# Opcja A: Przeskocz do konkretnej klatki (odkomentuj i ustaw numer)
# FRAME_ID = 200
# cap.set(cv2.CAP_PROP_POS_FRAMES, FRAME_ID)

# Opcja B: Pobierz kolejna klatke (domyslnie)
ret, frame = cap.read()

if not ret:
    print("Koniec wideo! Zrestartuj od Kroku 3 aby zaladowac ponownie.")
else:
    frame_rgb     = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    original_shape = frame.shape
    frame_idx      = int(cap.get(cv2.CAP_PROP_POS_FRAMES)) - 1
    timestamp_ms   = (frame_idx / fps) * 1000.0

    print(f"Klatka #{frame_idx} | {timestamp_ms:.0f} ms | Rozmiar: {original_shape}")

    plt.figure(figsize=(10, 6))
    plt.imshow(frame_rgb)
    plt.title(f"Klatka #{frame_idx} — {timestamp_ms:.0f} ms")
    plt.axis("off")
    plt.tight_layout()
    plt.show()

---
## Krok 5: Detekcja pilki (YOLO VB)

Identyczny kod jak w `engine.py` — mozna zmieniac `conf_threshold` i obserwowac wyniki na biezaco.

In [ ]:
input_ball = preprocess_yolo_input(frame_rgb)
ball_outs  = session_vb.run([output_name_vb], {input_name_vb: input_ball})
ball_boxes, ball_scores, _ = postprocess_yolo_output(
    ball_outs[0], original_shape, conf_threshold=CONF_BALL
)

print(f"Wykryte pilki: {len(ball_boxes)}")
for i, (box, score) in enumerate(zip(ball_boxes, ball_scores)):
    print(f"  Pilka #{i}: box={box}, score={score:.3f}")

# Wizualizacja
fig, ax = plt.subplots(1, figsize=(10, 6))
ax.imshow(frame_rgb)
for box, score in zip(ball_boxes, ball_scores):
    x1, y1, x2, y2 = box
    rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                               linewidth=2, edgecolor="yellow", facecolor="none")
    ax.add_patch(rect)
    ax.text(x1, y1-5, f"ball {score:.2f}", color="yellow", fontsize=9,
            bbox=dict(facecolor="black", alpha=0.5, pad=1))
ax.set_title("Detekcja pilki (YOLO VB)")
ax.axis("off")
plt.tight_layout()
plt.show()

---
## Krok 6: Detekcja osób (YOLO COCO)

Wykrywanie zawodnikow. Mozna testowac inne progi lub filtrowac klasy.

In [ ]:
input_coco = preprocess_yolo_input(frame_rgb)
coco_outs  = session_coco.run([output_name_coco], {input_name_coco: input_coco})
coco_boxes, coco_scores, coco_class_ids = postprocess_yolo_output(
    coco_outs[0], original_shape, conf_threshold=CONF_PERSON
)

# Filtruj tylko osoby (class_id == 0 to 'person' w COCO)
person_boxes  = [coco_boxes[i]  for i, cid in enumerate(coco_class_ids) if cid == 0]
person_scores = [coco_scores[i] for i, cid in enumerate(coco_class_ids) if cid == 0]

print(f"Wszystkie detekcje COCO:  {len(coco_boxes)}")
print(f"Wykryte osoby (class=0):  {len(person_boxes)}")

fig, ax = plt.subplots(1, figsize=(10, 6))
ax.imshow(frame_rgb)
for box, score in zip(person_boxes, person_scores):
    x1, y1, x2, y2 = box
    rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                               linewidth=2, edgecolor="cyan", facecolor="none")
    ax.add_patch(rect)
    ax.text(x1, y1-5, f"person {score:.2f}", color="cyan", fontsize=8,
            bbox=dict(facecolor="black", alpha=0.5, pad=1))

# Dorysuj pilki jesli byly wykryte
for box in ball_boxes:
    x1, y1, x2, y2 = box
    rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                               linewidth=2, edgecolor="yellow", facecolor="none")
    ax.add_patch(rect)

ax.set_title("Detekcja osob (cyan) i pilki (zolty)")
ax.axis("off")
plt.tight_layout()
plt.show()

---
## Krok 7: Najblizszy zawodnik do pilki

Algorytm wyboru gracza kontaktujacego sie z pilka — identyczny z engine.py.

In [ ]:
closest_person_box = None
detected_ball_box  = None

if len(ball_boxes) > 0 and len(person_boxes) > 0:
    ball_box_index   = np.argmax(ball_scores)
    ball_box         = ball_boxes[ball_box_index]
    detected_ball_box = ball_box

    min_dist = float("inf")
    for pbox in person_boxes:
        dist = get_distance_person_ball_np(np.array(pbox), np.array(ball_box))
        if dist < min_dist:
            min_dist = dist
            closest_person_box = pbox

    print(f"Najlepsza pilka: {ball_box}  (score={ball_scores[ball_box_index]:.3f})")
    print(f"Najblizszy gracz: {closest_person_box}  (dystans={min_dist:.1f} px)")

    fig, ax = plt.subplots(1, figsize=(10, 6))
    ax.imshow(frame_rgb)

    bx1, by1, bx2, by2 = ball_box
    ax.add_patch(patches.Rectangle((bx1, by1), bx2-bx1, by2-by1,
                  linewidth=3, edgecolor="yellow", facecolor="none"))

    px1, py1, px2, py2 = closest_person_box
    ax.add_patch(patches.Rectangle((px1, py1), px2-px1, py2-py1,
                  linewidth=3, edgecolor="red", facecolor="none"))
    ax.text(px1, py1-8, "FOCUS", color="red", fontsize=10, weight="bold",
            bbox=dict(facecolor="black", alpha=0.6, pad=2))

    ax.set_title("Wybrany gracz (czerwony) i pilka (zolty)")
    ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("Brak pilki lub osob na klatce — brak wyboru gracza.")

---
## Krok 8: Estymacja pozy MediaPipe

Wycinamy ROI gracza, skalujemy do kwadratu i uruchamiamy MediaPipe Pose.

In [ ]:
pose_features = None
pose_res = None

if closest_person_box is not None:
    px_min, py_min, px_max, py_max = closest_person_box
    px_min = max(0, int(px_min))
    py_min = max(0, int(py_min))
    px_max = min(original_shape[1], int(px_max))
    py_max = min(original_shape[0], int(py_max))

    if px_min < px_max and py_min < py_max:
        person_roi = frame_rgb[py_min:py_max, px_min:px_max]
        sq_frame, pad_l, pad_t = pad_frame_to_square(person_roi)
        pose_res = mp_pose.process(sq_frame)

        # Wizualizacja pozy na wycinku
        sq_annotated = sq_frame.copy()
        if pose_res.pose_landmarks:
            mp_drawing.draw_landmarks(
                sq_annotated,
                pose_res.pose_landmarks,
                mp_pose_solution.POSE_CONNECTIONS
            )
            print(f"[OK] Wykryto {len(pose_res.pose_landmarks.landmark)} punktow pozy")
        else:
            print("[!] MediaPipe nie wykryl pozy na tym kadrze")

        fig, axes = plt.subplots(1, 2, figsize=(10, 4))
        axes[0].imshow(person_roi)
        axes[0].set_title("ROI gracza")
        axes[0].axis("off")
        axes[1].imshow(sq_annotated)
        axes[1].set_title("MediaPipe Pose")
        axes[1].axis("off")
        plt.tight_layout()
        plt.show()
else:
    print("Brak wybranego gracza z Kroku 7.")

---
## Krok 9: Ekstrakcja cech i klasyfikacja akcji (RandomForest)

Normalizacja landmark'ow + wektor cech pilki => predykcja akcji.

In [ ]:
detected_action = "NONE"

if pose_res is not None and pose_res.pose_landmarks and detected_ball_box is not None:
    rel_landmarks = pose_res.pose_landmarks.landmark[11:25]  # tors + ramiona + biodra

    px_coords = [lm.x for lm in rel_landmarks]
    py_coords = [lm.y for lm in rel_landmarks]
    pm_x_min, pm_x_max = min(px_coords), max(px_coords)
    pm_y_min, pm_y_max = min(py_coords), max(py_coords)

    x_range = max(pm_x_max - pm_x_min, 1e-6)
    y_range = max(pm_y_max - pm_y_min, 1e-6)

    # 28 cech: 14 punktow x 2 wspolrzedne (znormalizowane)
    data = []
    for lm in rel_landmarks:
        data.append((lm.x - pm_x_min) / x_range)
        data.append((lm.y - pm_y_min) / y_range)

    # +3 cechy: pozycja i rozmiar pilki wzgledem gracza
    bx_min, by_min, bx_max, by_max = detected_ball_box
    data.append((bx_min - pm_x_min) / x_range)
    data.append((by_min - pm_y_min) / y_range)
    data.append(max((bx_max - bx_min) / x_range, (by_max - by_min) / y_range))

    print(f"Dlugosc wektora cech: {len(data)} (oczekiwane: 31)")

    if len(data) == 31:
        X = np.asarray(data).reshape(1, -1)
        pred  = rf_model.predict(X)
        proba = rf_model.predict_proba(X)
        detected_action = str(pred[0])

        print(f"\n>>> Wykryta akcja: {detected_action} <<<")
        print(f"Prawdopodobienstwa:")
        for cls, prob in zip(rf_model.classes_, proba[0]):
            bar = "#" * int(prob * 30)
            print(f"  {cls:15s}: {prob:.3f}  |{bar}")
    else:
        print(f"[!] Nieprawidlowy wektor cech: {len(data)} != 31")
else:
    print("Brak pozy lub pilki — nie mozna klasyfikowac akcji.")
    print(f"  pose_res.pose_landmarks: {pose_res.pose_landmarks if pose_res else None}")
    print(f"  detected_ball_box:       {detected_ball_box}")

---
## Krok 10: Podsumowanie klatki

Pelna wizualizacja wszystkich wykryc na jednej klatce.

In [ ]:
fig, ax = plt.subplots(1, figsize=(12, 7))
ax.imshow(frame_rgb)

# Wszystkie osoby (szare)
for box in person_boxes:
    x1, y1, x2, y2 = box
    ax.add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1,
                  linewidth=1, edgecolor="gray", facecolor="none", linestyle="--"))

# Pilka (zolta)
for box in ball_boxes:
    x1, y1, x2, y2 = box
    ax.add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1,
                  linewidth=3, edgecolor="yellow", facecolor="none"))
    ax.text(x1, y2+15, "BALL", color="yellow", fontsize=9, weight="bold")

# Fokus gracz (czerwony + akcja)
if closest_person_box is not None:
    px1, py1, px2, py2 = closest_person_box
    color = "lime" if detected_action != "NONE" else "red"
    ax.add_patch(patches.Rectangle((px1, py1), px2-px1, py2-py1,
                  linewidth=3, edgecolor=color, facecolor="none"))
    label = detected_action if detected_action != "NONE" else "FOCUS"
    ax.text(px1, py1-10, label, color=color, fontsize=12, weight="bold",
            bbox=dict(facecolor="black", alpha=0.7, pad=3))

ax.set_title(f"Klatka #{frame_idx} | Akcja: {detected_action}", fontsize=13)
ax.axis("off")
plt.tight_layout()
plt.show()
print(f"\nWynik analizy klatki #{frame_idx}: action={detected_action}")

---
## Narzedzia dodatkowe

### Analiza wielu klatek z proba detekcji
Testuje N kolejnych klatek i wypisuje wyniki — przydatne do sprawdzenia sporadycznych detekcji.

In [ ]:
SAMPLE_EVERY_N_FRAMES = 30  # co ktora klatke pobierac
MAX_SAMPLES = 20            # maksymalna liczba prob

# Resetuj wideo
cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

results_batch = []
f_idx = 0
sampled = 0

while cap.isOpened() and sampled < MAX_SAMPLES:
    ret, frame = cap.read()
    if not ret:
        break
    if f_idx % SAMPLE_EVERY_N_FRAMES != 0:
        f_idx += 1
        continue

    f_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    ts_ms = (f_idx / fps) * 1000.0

    inp_b = preprocess_yolo_input(f_rgb)
    b_out = session_vb.run([output_name_vb], {input_name_vb: inp_b})
    b_boxes, b_scores, _ = postprocess_yolo_output(b_out[0], frame.shape, conf_threshold=CONF_BALL)

    has_ball = len(b_boxes) > 0
    action   = "NO_BALL"

    if has_ball:
        inp_c = preprocess_yolo_input(f_rgb)
        c_out = session_coco.run([output_name_coco], {input_name_coco: inp_c})
        c_boxes, c_scores, c_ids = postprocess_yolo_output(c_out[0], frame.shape, conf_threshold=CONF_PERSON)
        p_boxes = [c_boxes[i] for i, cid in enumerate(c_ids) if cid == 0]
        action = "BALL_NO_PERSON" if len(p_boxes) == 0 else "PROCESSING"

    results_batch.append({"frame": f_idx, "ts_ms": ts_ms, "ball": has_ball, "action": action})
    f_idx += 1
    sampled += 1

print(f"{'Klatka':>8} {'Czas[ms]':>10} {'Pilka':>7} {'Akcja':<15}")
print("-" * 48)
for r in results_batch:
    ball_str = "TAK" if r["ball"] else "NIE"
    print(f"{r['frame']:>8} {r['ts_ms']:>10.0f} {ball_str:>7} {r['action']:<15}")

---
## Zamkniecie wideo (opcjonalne)
Uruchom po zakonczeniu pracy aby zwolnic zasoby.

In [ ]:
cap.release()
print("Wideo zamkniete.")